In [ ]:
# Настройка путей 
import sys
from pathlib import Path

# Получаем путь к корню проекта: поднимаемся из "Jupyter Notebooks/" на уровень выше
project_root = Path().resolve().parent

# Добавляем корень проекта в sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added to sys.path: {project_root}")

#### Импорт библиотек и загрузка данных

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from config import RAW_DATA_DIR
from src.db.queries import run_query

# Загрузка данных
df = pd.read_csv(RAW_DATA_DIR / "third_wave_coffee_shop.csv", parse_dates=['datetime', 'sale_date'])


#### 2. Проверка качества данных (пропуски, дубликаты)

In [8]:
# Пропущенные значения
print("Пропущенные значения:")
print(df.isnull().sum())

# Дубликаты (полные строки)
print(f"\nКоличество дубликатов: {df.duplicated().sum()}")

# Удаление дубликатов, если нужно (можно раскомментировать)
# df = df.drop_duplicates()

# Проверка уникальных значений в ключевых колонках
print("\nУникальные напитки:")
print(df['coffee_name'].value_counts().head(10))

Пропущенные значения:
transaction_id    0
customer_id       0
sale_date         0
sale_time         0
day_of_week       0
day_name          0
is_weekend        0
coffee_name       0
drink_price       0
total_cost        0
payment_method    0
datetime          0
time_of_day       0
month             0
hour              0
date              0
dtype: int64

Количество дубликатов: 276

Уникальные напитки:
coffee_name
Cappuccino              1165
Latte                   1042
Batch brew               844
Latte with add           743
Raf coffee               342
Flat white               335
Warming drinks (tea)     267
Espresso                 205
Name: count, dtype: int64


#### 3. Преобразование дат (для временного анализа)

In [3]:
# Добавление месяца и часа (если нужно для группировки)
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['date'] = df['datetime'].dt.date
display(df[['datetime', 'month', 'hour', 'date']].head())

,datetime,month,hour,date
0,2025-07-01 08:24:55,7,8,2025-07-01
1,2025-07-01 08:24:55,7,8,2025-07-01
2,2025-07-01 08:42:32,7,8,2025-07-01
3,2025-07-01 08:43:55,7,8,2025-07-01
4,2025-07-01 08:44:55,7,8,2025-07-01


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4943 entries, 0 to 4942
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  4943 non-null   object        
 1   customer_id     4943 non-null   object        
 2   sale_date       4943 non-null   datetime64[ns]
 3   sale_time       4943 non-null   object        
 4   day_of_week     4943 non-null   int64         
 5   day_name        4943 non-null   object        
 6   is_weekend      4943 non-null   bool          
 7   coffee_name     4943 non-null   object        
 8   drink_price     4943 non-null   float64       
 9   total_cost      4943 non-null   float64       
 10  payment_method  4943 non-null   object        
 11  datetime        4943 non-null   datetime64[ns]
 12  time_of_day     4943 non-null   object        
 13  month           4943 non-null   int32         
 14  hour            4943 non-null   int32         
 15  date

In [6]:
print(df.columns.tolist())

['transaction_id', 'customer_id', 'sale_date', 'sale_time', 'day_of_week', 'day_name', 'is_weekend', 'coffee_name', 'drink_price', 'total_cost', 'payment_method', 'datetime', 'time_of_day', 'month', 'hour', 'date']


#### Структура данных:

- **Каждая строка = 1 напиток**

- **Каждая транзакция (`transaction_id`) = 1 заказ (чек)**

- `total_cost` — **одинаков** для всех строк одной транзакции

- **Все расчёты по чекам, выручке, времени, дням — только по уникальным `transaction_id`**

In [7]:
# Создание df_tx — только уникальные транзакции
df_tx = df.drop_duplicates(subset='transaction_id').reset_index(drop=True)

# Добавляем количество напитков в заказе
df_tx['num_drinks'] = df.groupby('transaction_id').size().values

print(f"Уникальных транзакций: {len(df_tx)}")
print(f"Всего строк в df: {len(df)}")
print(f"Среднее количество напитков в заказе: {df_tx['num_drinks'].mean():.2f}")

Уникальных транзакций: 3547
Всего строк в df: 4943
Среднее количество напитков в заказе: 1.39
